# Train TrOCR on IAM Handwriting Dataset

This notebook fine-tunes the Microsoft TrOCR model on the IAM Handwriting dataset using Hugging Face datasets (`Teklia/IAM-line`).

### ⚠️ IMPORTANT: Enable GPU ⚠️
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU NOT Detected! Please change runtime type to GPU.")
    # raise RuntimeError("No GPU found. Training will be too slow.")

In [ ]:
# MOUNT GOOGLE DRIVE (MANDATORY)
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully.")

In [ ]:
# 1. Clone or Update the repository
import os

if os.path.exists('handwriting_recog'):
    %cd handwriting_recog
    !git pull origin main
else:
    !git clone https://github.com/Bhuvan-018/handwriting_recog
    %cd handwriting_recog

In [ ]:
# 2. Install dependencies
!pip install -r requirements.txt
!pip install huggingface_hub

In [ ]:
# 3. Run the training script (SKIP THIS since you have the model)
# !python train_hf.py

In [ ]:
# 4. Restore Model and Deploy to MODEL REPOSITORY (Unlimited Storage)
import os
import shutil
import glob
from huggingface_hub import HfApi, login, create_repo

# --- Configuration ---
HF_TOKEN = "YOUR_HF_WRITE_TOKEN" # @param {type:"string"}
MODEL_REPO_ID = "bhuvan-018/trocr-handwriting-iam" # @param {type:"string"}

# Clean inputs
HF_TOKEN = HF_TOKEN.strip()
MODEL_REPO_ID = MODEL_REPO_ID.strip()

MODEL_EXTRACT_DIR = "models/trocr_finetuned_iam_hf"
ZIP_FILE_PATH = "/content/drive/MyDrive/Handwriting_Model_Backup/trocr_model_20260301_1159.zip"

if HF_TOKEN == "YOUR_HF_WRITE_TOKEN" or not HF_TOKEN:
    print("⚠️ Please enter your Hugging Face Write Token above!")
else:
    try:
        login(token=HF_TOKEN)
        print("✅ Authenticated with Hugging Face.")
        api = HfApi()
        
        # --- Unzip Model ---
        if os.path.exists(ZIP_FILE_PATH):
            if not os.path.exists(MODEL_EXTRACT_DIR):
                print(f"📦 Unzipping {ZIP_FILE_PATH}...")
                !unzip -o "{ZIP_FILE_PATH}" -d .
            else:
                print(f"✅ Model already extracted at {MODEL_EXTRACT_DIR}")
            
            # Verify path
            final_model_dir = MODEL_EXTRACT_DIR
            if not os.path.exists(MODEL_EXTRACT_DIR):
                 if os.path.exists("trocr_finetuned_iam_hf"):
                      final_model_dir = "trocr_finetuned_iam_hf"
                 elif os.path.exists("models/trocr_finetuned_iam_hf"):
                      final_model_dir = "models/trocr_finetuned_iam_hf"
            
            print(f"✅ Ready to deploy model from: {final_model_dir}")
            
            # --- Clean Checkpoints ---
            print("🧹 Cleaning up intermediate checkpoints...")
            checkpoints = [d for d in os.listdir(final_model_dir) if d.startswith('checkpoint-')]
            for ckpt in checkpoints:
                shutil.rmtree(os.path.join(final_model_dir, ckpt))

            # --- Create Model Repo ---
            print(f"🔨 Creating/Checking Model Repository: {MODEL_REPO_ID}...")
            create_repo(repo_id=MODEL_REPO_ID, repo_type="model", exist_ok=True)
            
            # --- Upload to Model Repo ---
            print("🚀 Uploading to Hugging Face Model Hub (Unlimited Storage)...")
            api.upload_folder(
                folder_path=final_model_dir,
                repo_id=MODEL_REPO_ID,
                repo_type="model",
                commit_message="Initial model upload",
                ignore_patterns=[".git", ".ipynb_checkpoints"]
            )
            
            print("✅ Successfully deployed Model!")
            print(f"🔗 Model Link: https://huggingface.co/{MODEL_REPO_ID}")
            print("\n👉 NOTE: To use this model in your Space, update the 'model_id' in your app.py to point to this new link!")
            
        else:
            print(f"❌ Error: Zip file not found at {ZIP_FILE_PATH}")
            
    except Exception as e:
        print(f"❌ Deployment failed: {e}")